# Agent Network Lab
## 25 clients + 7 network agents · One Qwen3-0.6B · vLLM Metal
Run on your Apple Silicon Mac. Use this folder’s `.venv/bin/python` kernel. All app source is included below for editing in Jupyter or VS Code. **Do not load a model inside this notebook:** every agent calls the one external vLLM server.

The network is simulated; model inference is real. Read README.md for scope and timing semantics.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT = Path.cwd()
if not (ROOT / "manage.py").exists():
    ROOT = Path("/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab")
os.chdir(ROOT)
print("Project:", ROOT)
print("Kernel:", sys.executable)

Project: /Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab
Kernel: /Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/.venv/bin/python


### 1. Launch services
Idempotent launcher: one owned model server plus one dashboard server. First model download may take a few minutes. Inspect the log if readiness fails.

In [2]:
result = subprocess.run([str(ROOT/".venv/bin/python"), "manage.py", "start"], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

Dashboard: http://127.0.0.1:8020
Model loading/download log: /Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/logs/model.log
{
  "model": {
    "pid": 11153,
    "command": [
      "/opt/homebrew/bin/vllm",
      "serve",
      "mlx-community/Qwen3-0.6B-4bit",
      "--revision",
      "73e3e38d981303bc594367cd910ea6eb48349da8",
      "--host",
      "127.0.0.1",
      "--port",
      "8011",
      "--served-model-name",
      "qwen3-network",
      "--max-model-len",
      "2048",
      "--max-num-seqs",
      "20",
      "--kv-cache-memory-bytes",
      "1073741824",
      "--gpu-memory-utilization",
      "0.15",
      "--generation-config",
      "vllm"
    ],
    "started": 1793590951.087738,
    "alive": true
  },
  "app": {
    "pid": 11154,
    "command": [
      "/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/.venv/bin/python",
      "-m",
      "uvicorn",
      "server:app",
      "--host",
      "127.0.0.1",
      "--port",
   

### 2. Readiness
This bounded check does not start another model. Rerun if the initial download is still in progress.

In [3]:
import httpx
with httpx.Client(trust_env=False, timeout=5) as client:
    for attempt in range(12):
        try:
            health = client.get("http://127.0.0.1:8020/api/health").json()
            if health["ready"]:
                print("One model ready:", client.get("http://127.0.0.1:8011/v1/models").json())
                break
        except Exception as exc:
            health = {"error": str(exc)}
        time.sleep(2)
    else:
        print("Not ready yet:", health)
        print((ROOT/"logs/model.log").read_text(errors="replace")[-6000:])

One model ready: {'object': 'list', 'data': [{'id': 'qwen3-network', 'object': 'model', 'created': 1793594136, 'owned_by': 'vllm', 'root': 'mlx-community/Qwen3-0.6B-4bit', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-ad6e7128961b36ce', 'object': 'model_permission', 'created': 1793594136, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


### 3. Open the central app
Start/pause, select agents, fail links or services, adjust traffic and inspect conversations. Keep this tab open rather than loading another model in the notebook.

In [4]:
from IPython.display import display, HTML
display(HTML('<a href="http://127.0.0.1:8020" target="_blank">Open Agent Network Lab ↗</a>'))

### 4. Start and observe
All 32 logical agents share the same inference gateway. Timing values distinguish virtual network time from wall-clock inference time.

In [5]:
with httpx.Client(trust_env=False, timeout=10) as client:
    response = client.post("http://127.0.0.1:8020/api/control/start")
    response.raise_for_status()
    print(response.json()["metrics"])

{'server_rss_mb': 63, 'invalid': 1, 'generated': 0, 'completed': 0, 'dropped': 0, 'inflight': 1, 'tokens': 65352, 'valid': 100, 'fallbacks': 0, 'network_latency': 0, 'inference_latency': 0.664, 'queue_latency': 0.004}


In [6]:
with httpx.Client(trust_env=False, timeout=10) as client:
    state = client.get("http://127.0.0.1:8020/api/state").json()
print(json.dumps({"metrics": state["metrics"], "active": state["active"], "pending": state["pending"], "agents": [{"id": a["id"], "decisions": a["decisions"], "last": a["last_decision"]} for a in state["agents"]]}, indent=2))

{
  "metrics": {
    "server_rss_mb": 63,
    "invalid": 1,
    "generated": 0,
    "completed": 0,
    "dropped": 0,
    "inflight": 1,
    "tokens": 65352,
    "valid": 100,
    "fallbacks": 0,
    "network_latency": 0,
    "inference_latency": 0.664,
    "queue_latency": 0.004
  },
  "active": [],
  "pending": [],
  "agents": [
    {
      "id": "C01",
      "decisions": 1,
      "last": {
        "action": "debate_message",
        "target": "C06",
        "reason": "Surface codes offer a promising path for fault-tolerant quantum computing, but their memory limitations (e.g., distance and overhead) may hinder their universal fault-tolerant capabilities. This is a bounded claim with uncertainty about their practical implementation.",
        "argument": "Surface codes offer a promising path for fault-tolerant quantum computing, but their memory limitations (e.g., distance and overhead) may hinder their universal fault-tolerant capabilities. This is a bounded claim with uncertainty a

### 5. Traffic experiment
Raise service time and load, or choose inference concurrency 1 versus 2. Changes affect the simulator, not macOS networking. Use a fresh run and warm model for fair comparisons.

In [7]:
settings = dict(load=2, latency=100, bandwidth=32, loss=0, service_delay=2.0, concurrency=1)
with httpx.Client(trust_env=False) as client:
    response = client.post("http://127.0.0.1:8020/api/config", json=settings)
    response.raise_for_status()
    print(response.json())

{'load': 2, 'latency': 100, 'bandwidth': 32, 'loss': 0, 'service_delay': 2.0, 'concurrency': 1}


### 6. Fail or restore A8
This toggles availability. Check the returned node state before toggling again.

In [8]:
with httpx.Client(trust_env=False) as client:
    state = client.post("http://127.0.0.1:8020/api/toggle/agent/A8").json()
print(next(a for a in state["agents"] if a["id"] == "A8"))

{'id': 'A8', 'role': 'Service', 'up': False, 'sent': 0, 'received': 0, 'decisions': 0, 'status': 'idle', 'preferred_service': 'A8', 'preferred_hop': '', 'service_free': 0, 'last_decision': {}, 'inbox': [], 'service_queue': 0}


### 7. Pause and export
Export captures the event log; replay is recorded playback, not deterministic model regeneration.

In [9]:
with httpx.Client(trust_env=False) as client:
    client.post("http://127.0.0.1:8020/api/control/pause").raise_for_status()
    recording = client.get("http://127.0.0.1:8020/api/export").json()
path = ROOT / "data" / f"run-{recording["run"]}.json"
path.write_text(json.dumps(recording, indent=2))
print(path)

/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/data/run-2e672c4d.json


### Scientific debate: quantum fault tolerance
25 participants in five teams share the same tiny model. Four rounds cover opening claims, cross-examination, rebuttal and synthesis. Source IDs are supplied evidence references, not fact checking. The transcript is centrally shared; messages are also routed through the simulator. The debate stops after 100 turns.

In [10]:
with httpx.Client(trust_env=False, timeout=15) as client:
    client.post("http://127.0.0.1:8020/api/mode/debate").raise_for_status()
    client.post("http://127.0.0.1:8020/api/control/start").raise_for_status()
print("Scientific debate started. Follow it in the dashboard.")

Scientific debate started. Follow it in the dashboard.


### Inspect a full trace
Traces expire three hours after creation in wall-clock time. API reads hide expired records immediately. Physical cleanup runs every 60 seconds while the app is alive and on restart. Downloaded exports and manual notebook outputs are not covered by deletion. Do not save raw notebook outputs if you want to avoid independent copies.

In [11]:
with httpx.Client(trust_env=False) as client:
    traces = client.get("http://127.0.0.1:8020/api/traces?kind=debate").json()["items"]
    if traces:
        trace = client.get("http://127.0.0.1:8020/api/traces/" + traces[0]["id"]).json()
        print({"id": trace["id"], "expires": trace["expires"], "stages": [s["stage"] for s in trace["payload"]["spans"]]})
    else:
        print("No debate trace yet; rerun after a participant speaks.")

{'id': '6117f84f7ae9410c', 'expires': 1793601884.170676, 'stages': ['inference_request', 'inference_response', 'validated']}


### 8. Shutdown and unload the model
This stops both project-owned process groups. No background inference is left running by this launcher. Model files remain cached for next time.

In [12]:
subprocess.run([str(ROOT/".venv/bin/python"), "manage.py", "stop"], check=True)

Stopped project-owned app and inference server.


CompletedProcess(args=['/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/.venv/bin/python', 'manage.py', 'stop'], returncode=0)

## Editable app source
These cells reproduce the accompanying source. Run only when you want to save edits, and restart services afterwards. They do not load any model. The normal launch cells above work without running these source cells.

### engine.py

In [13]:
%%writefile engine.py
"""Twenty-five clients and seven network actors, one inference gateway, an explicitly simulated network."""
import asyncio, heapq, json, random, sqlite3, time, uuid
from collections import deque
from dataclasses import dataclass, field, asdict
from pathlib import Path
import httpx
import psutil
from traces import TraceStore, TTL
from debate import Debate

ROOT = Path(__file__).resolve().parent
CLIENTS = [f'C{i:02d}' for i in range(1,26)]
GATEWAYS = {client: ('A4' if i < 13 else 'A5') for i,client in enumerate(CLIENTS)}
ROLES = {**{client:'Client' for client in CLIENTS}, **dict(zip(['A4','A5','A6','A7','A8','A9','A10'],['Gateway','Gateway','Router','Router','Service','Service','Observer']))}
PAIRS = list(GATEWAYS.items()) + [(f'A{a}',f'A{b}') for a,b in [(4,6),(4,7),(5,6),(5,7),(6,8),(6,9),(7,8),(7,9),(10,4),(10,5),(10,6),(10,7),(10,8),(10,9)]]

@dataclass
class Agent:
    id: str
    role: str
    up: bool = True
    sent: int = 0
    received: int = 0
    decisions: int = 0
    status: str = 'idle'
    preferred_service: str = 'A8'
    preferred_hop: str = ''
    service_free: float = 0
    last_decision: dict = field(default_factory=dict)
    inbox: deque = field(default_factory=lambda: deque(maxlen=6))

class Lab:
    def __init__(self, db_path=None):
        ROOT.joinpath('data').mkdir(exist_ok=True)
        self.db = sqlite3.connect(db_path or ROOT/'data/events.sqlite')
        self.db.execute('PRAGMA journal_mode=WAL')
        self.db.execute('CREATE TABLE IF NOT EXISTS events (seq INTEGER PRIMARY KEY, run TEXT, wall REAL, sim REAL, kind TEXT, payload TEXT)')
        self.db.commit()
        self.traces=TraceStore(self.db)
        self.mode="network"
        self.client = httpx.AsyncClient(timeout=90, trust_env=False)
        self.model_url = 'http://127.0.0.1:8011/v1'
        self.model = 'qwen3-network'
        self.model_health = 'not checked'
        self.model_error = ''
        self.running = False
        self.pending = {}
        self.active = {}
        self.results = deque()
        self.workers = []
        self.reset()
        self.debate=Debate(self)

    def reset(self):
        self.running = False
        self.epoch = getattr(self,'epoch',0)+1
        self.pending.clear(); self.results.clear()
        if hasattr(self,'debate'):self.debate.reset()
        self.run_id = str(uuid.uuid4())[:8]
        self.clock = 0.; self.rng = random.Random(42)
        self.agents = {aid:Agent(aid,role) for aid,role in ROLES.items()}
        self.links = {f'{a}-{b}':dict(id=f'{a}-{b}',a=a,b=b,up=True,free={},sent=0) for a,b in PAIRS}
        self.config = dict(load=1, latency=100, bandwidth=32, loss=0, service_delay=.2, concurrency=20)
        self.events = deque(maxlen=160); self.inflight=[]; self.seq=0
        self.generated=0; self.completed=0; self.dropped=0; self.tokens=0; self.valid=0; self.invalid=0; self.failures=0
        self.latencies=deque(maxlen=1000); self.inference_times=deque(maxlen=200); self.queue_times=deque(maxlen=200)
        self.log('run','New run; simulated clock paused. All agents share one inference endpoint.')

    def log(self,kind,text,**extra):
        event=dict(kind=kind,text=text,sim=round(self.clock,3),wall=time.time(),**extra)
        self.events.append(event)
        self.db.execute('INSERT INTO events(run,wall,sim,kind,payload) VALUES(?,?,?,?,?)',(self.run_id,event['wall'],self.clock,kind,json.dumps(event)))
        self.db.commit()

    def neighbors(self,aid):
        return [l['b'] if l['a']==aid else l['a'] for l in self.links.values() if aid in (l['a'],l['b']) and l['up'] and self.agents[l['b'] if l['a']==aid else l['a']].up]

    def next_hop(self,source,target,visited):
        queue=deque([(source,[])])
        seen=set(visited)-{source}; seen.add(source)
        while queue:
            node,path=queue.popleft()
            if node==target:return path[0] if path else None
            for peer in self.neighbors(node):
                if peer=='A10' and target!='A10':continue
                if peer not in seen:seen.add(peer);queue.append((peer,path+[peer]))
        return None

    def send(self,source,target,kind='status',request=None,visited=None,payload=None):
        visited=list(visited or [])
        if payload is None and kind=='status':payload={'agent':source,'up':self.agents[source].up,'service_queue':round(max(0,self.agents[source].service_free-self.clock),2),'note':str(self.agents[source].last_decision.get('reason','Available'))[:120]}
        if payload is None and request and kind in ('request','response'):
            payload={'note':json.dumps(request.get('task') if kind=='request' else request.get('response')),'trace_id':request.get('trace_id')}
        if not visited and kind in ('status','debate','request','response') and payload:
            payload=dict(payload)
            payload['_message_trace']=self.traces.create(self.run_id,'message',source,title=f'{source} → {target} · {kind}',source=source,recipient=target,message_kind=kind,text=payload.get('argument',payload.get('note','')),reply_to=payload.get('reply_to'),evidence_id=payload.get('evidence_id'),uncertainty=payload.get('uncertainty'),model_trace_id=payload.get('trace_id'),status='in_transit')
        if payload and payload.get('_message_trace'):self.traces.append(payload['_message_trace'],'hop',sim=self.clock,source=source,target=target)
        if request and request.get('trace_id'):self.traces.append(request['trace_id'],'route',sim=self.clock,source=source,target=target,message_kind=kind)
        if not self.agents[source].up or not self.agents[target].up:
            self.drop(source,target,'node unavailable',request,payload);return
        if source==target:self.arrive(source,target,kind,request,visited,payload);return
        hop=self.next_hop(source,target,visited)
        preferred=self.agents[source].preferred_hop
        # A suggested hop is used only if it has a loop-free path to this target.
        if preferred in self.neighbors(source) and preferred not in visited:
            if preferred==target or self.next_hop(preferred,target,visited+[source]):hop=preferred
        if not hop:self.drop(source,target,'no reachable route',request,payload);return
        link=next(l for l in self.links.values() if {l['a'],l['b']}=={source,hop})
        direction=source+'>'+hop
        start=max(self.clock,link['free'].get(direction,0))
        if start-self.clock>8:self.drop(source,target,'link queue full',request,payload);return
        size=4 if kind in ('request','response') else .5 # simulated KiB, not actual HTTP bytes
        finish=start+size/self.config['bandwidth'];link['free'][direction]=finish
        due=finish+self.config['latency']/1000
        self.seq+=1
        heapq.heappush(self.inflight,(due,self.seq,dict(source=source,hop=hop,target=target,kind=kind,request=request,visited=visited+[source],link=link['id'],payload=payload)))
        if request and request.get('trace_id'):self.traces.append(request['trace_id'],'link_queued',sim=self.clock,source=source,hop=hop,link=link['id'],virtual_queue_seconds=start-self.clock,virtual_due=due,message_kind=kind)
        self.agents[source].sent+=1;link['sent']+=1
        self.log('send',f'{source} → {hop}: {kind}'+(' · '+payload.get('note','') if payload else ''),source=source,target=hop,final=target,payload=payload)

    def drop(self,source,target,reason,request=None,payload=None):
        if payload and payload.get("_message_trace"):self.traces.append(payload["_message_trace"],"dropped",status="dropped",sim=self.clock,reason=reason)
        if request and request.get("trace_id"):self.traces.append(request["trace_id"],"dropped",status="dropped",sim=self.clock,source=source,target=target,reason=reason)
        self.dropped+=1;self.log('drop',f'{source} → {target}: {reason}',source=source,target=target)

    def arrive(self,aid,target,kind,request,visited,payload=None):
        agent=self.agents[aid];agent.received+=1
        if request and request.get('trace_id'):self.traces.append(request['trace_id'],'delivered',sim=self.clock,agent=aid,message_kind=kind)
        agent.inbox.append(dict(kind=kind,source=visited[-1] if visited else aid,time=round(self.clock,2),wall=time.time(),payload=payload))
        if aid!=target:
            if kind=='request' and agent.role=='Gateway':
                preferred=agent.preferred_service
                if self.agents[preferred].up:target=preferred
            self.send(aid,target,kind,request,visited,payload);return
        if payload and payload.get('_message_trace'):self.traces.append(payload['_message_trace'],'delivered',status='delivered',sim=self.clock,recipient=aid)
        if kind=='request' and agent.role=='Service':
            start=max(agent.service_free,self.clock)
            if start-self.clock>12:self.drop(aid,request['origin'],'service queue full',request);return
            agent.service_free=start+self.config['service_delay']
            if request.get('trace_id'):self.traces.append(request['trace_id'],'service_queued',status='processing',sim=self.clock,service=aid,start=start,finish=agent.service_free,task=request.get('task'))
            self.seq+=1
            heapq.heappush(self.inflight,(agent.service_free,self.seq,dict(kind='service_complete',source=aid,request=request)))
        elif kind=='response':
            self.completed+=1;latency=self.clock-request['created'];self.latencies.append(latency)
            if request.get('trace_id'):self.traces.append(request['trace_id'],'response_received',status='complete',sim=self.clock,agent=aid,result=request.get('response'),virtual_latency=latency)
            self.log('complete',f"{request['id']} completed in {latency:.2f} simulated seconds",source=aid)

    def observation(self,aid):
        a=self.agents[aid]
        return dict(agent=aid,role=a.role,neighbors=self.neighbors(aid),services={x:dict(up=self.agents[x].up,queue_delay=round(max(0,self.agents[x].service_free-self.clock),2)) for x in ['A8','A9']},recent=list(a.inbox)[-3:],allowed_actions=['report_status','wait']+(['set_service'] if a.role=='Gateway' else [])+(['set_next_hop'] if a.role=='Router' else []))

    def enqueue(self,aid):
        if aid not in self.pending and aid not in self.active and self.agents[aid].up:
            self.pending[aid]=(time.monotonic(),self.epoch,self.observation(aid));self.agents[aid].status='queued'

    async def health(self):
        try:
            r=await self.client.get(self.model_url+'/models');r.raise_for_status()
            names=[m['id'] for m in r.json()['data']]
            if names!=[self.model]:raise ValueError(f'Expected only {self.model}, got {names}')
            self.model_health='ready';self.model_error=''
        except Exception as e:self.model_health='offline';self.model_error=str(e)[:300]
        return self.model_health=='ready'

    async def infer(self,aid,obs,queue_wait=0):
        healthy=[key for key,value in obs['services'].items() if value['up']]
        schema={'type':'object','properties':{'action':{'type':'string','enum':obs['allowed_actions']},'target':{'type':'string','enum':list(dict.fromkeys(['']+obs['neighbors']+healthy))},'reason':{'type':'string'}},'required':['action','target','reason'],'additionalProperties':False}
        messages=[{'role':'system','content':'You are a network simulation agent. Choose one allowed action from the observation. Gateways should set_service to a healthy service with the lower queue_delay. Other roles should normally report_status to share observations with neighbors; use wait only when no useful action is available. A service with up=true is available. Reply only with compact JSON: action, target, reason. Use target "" for report_status or wait. Keep reason under 12 words.'},{'role':'user','content':json.dumps(obs)}]
        trace=self.traces.create(self.run_id,'inference',aid,title=f'Network decision · {aid}',queue_seconds=queue_wait,observation=obs)
        error=''
        for attempt in range(2):
            try:
                request_body=dict(model=self.model,messages=messages,max_tokens=128,temperature=0,chat_template_kwargs={'enable_thinking':False},response_format={'type':'json_schema','json_schema':{'name':'network_action','strict':True,'schema':schema}})
                self.traces.append(trace,'inference_request',attempt=attempt+1,request=request_body)
                r=await self.client.post(self.model_url+'/chat/completions',json=request_body)
                self.traces.append(trace,'http_response',attempt=attempt+1,status_code=r.status_code,body=r.text)
                r.raise_for_status();body=r.json();self.tokens+=body.get('usage',{}).get('total_tokens',0)
                decision=json.loads(body['choices'][0]['message']['content'])
                action=decision.get('action');target=decision.get('target','')
                if action not in obs['allowed_actions']:raise ValueError('Action not allowed for this role')
                if action=='set_service' and target not in healthy:raise ValueError('Service is unavailable')
                if action=='set_next_hop' and target not in obs['neighbors']:raise ValueError('Not an adjacent peer')
                self.traces.append(trace,'validated',status='validated',result=decision)
                return decision,None
            except (httpx.HTTPError,ValueError,KeyError,TypeError) as e:
                error=str(e)[:220]
                self.traces.append(trace,'error',error=error,attempt=attempt+1)
                if not isinstance(e,httpx.HTTPError):self.invalid+=1
                if isinstance(e,httpx.HTTPError):break
                messages.append({'role':'user','content':'Invalid response. Return a valid allowed action with action, target, reason.'})
        self.traces.append(trace,'fallback',status='fallback',error=error)
        return {'action':'report_status','target':'','reason':'Deterministic fallback: inference unavailable or invalid.'},error

    async def worker(self,index):
        while True:
            if not self.pending or not self.running or index>=self.config['concurrency']:
                await asyncio.sleep(.1);continue
            aid=next(iter(self.pending));queued,epoch,obs=self.pending.pop(aid)
            self.active[aid]=time.monotonic();self.agents[aid].status='thinking'
            wait=time.monotonic()-queued;started=time.monotonic()
            try:
                result,error=await (self.debate.respond(aid,obs,wait) if obs.get('mode')=='debate' else self.infer(aid,obs,wait))
                result['_completed_wall']=time.time()
                self.results.append((epoch,aid,result,error,wait,time.monotonic()-started))
                if epoch==self.epoch:self.agents[aid].status='buffered'
            finally:self.active.pop(aid,None)

    def apply_results(self):
        while self.results:
            epoch,aid,result,error,wait,duration=self.results.popleft()
            if epoch!=self.epoch:continue
            a=self.agents[aid];a.status='idle'
            if not a.up:
                if result.get('action')=='debate_message':self.debate.outstanding=False;self.debate.turn+=1
                continue
            if (result['action']=='set_service' and not self.agents[result['target']].up) or (result['action']=='set_next_hop' and result['target'] not in self.neighbors(aid)):
                self.log('stale',f'{aid}: topology changed before decision applied; discarded.',source=aid)
                continue
            a.last_decision={**result,'fallback':bool(error)};a.decisions+=1
            self.queue_times.append(wait);self.inference_times.append(duration)
            if error:self.failures+=1
            else:self.valid+=1
            action=result['action'];target=result.get('target','')
            if action=='debate_message':
                self.debate.apply(aid,result,error)
                continue
            if action=='set_service' and self.agents[target].up:a.preferred_service=target
            elif action=='set_next_hop' and target in self.neighbors(aid):a.preferred_hop=target
            elif action=='report_status':
                for peer in self.neighbors(aid):self.send(aid,peer,'status')
            self.log('fallback' if error else 'decision',f"{aid}: {action} {target} · {result.get('reason','')}",source=aid,error=error)

    def step(self):
        self.apply_results();self.clock+=.5
        if self.mode=='network' and int(self.clock*2)%6==0:
            for aid in CLIENTS:
                for _ in range(self.config['load']):
                    if self.agents[aid].up:
                        self.generated+=1
                        task={'operation':'sum','numbers':[int(aid[1:]),self.generated%10]} if self.generated%3==0 else {'operation':'uppercase','text':f'hello from {aid}'} if self.generated%3==1 else {'operation':'health_check'}
                        trace=self.traces.create(self.run_id,'request',aid,title=f'R{self.generated} · {task["operation"]}',task=task)
                        request=dict(id=f'R{self.generated}',origin=aid,created=self.clock,trace_id=trace,task=task,wall=time.time())
                        gateway=GATEWAYS[aid]
                        target=self.agents[gateway].preferred_service
                        if not self.agents[target].up:target='A9' if target=='A8' else 'A8'
                        self.send(aid,target,'request',request)
        processed=0
        while self.inflight and self.inflight[0][0]<=self.clock and processed<500:
            due,_,event=heapq.heappop(self.inflight);processed+=1
            # Quantized virtual time: deliveries are processed on the next 0.5s tick.
            if event['kind']=='service_complete':
                if self.agents[event['source']].up:
                    req=event['request'];task=req.get('task',{'operation':'health_check'})
                    value=sum(task['numbers']) if task['operation']=='sum' else task['text'].upper() if task['operation']=='uppercase' else 'healthy'
                    req['response']={'service':event['source'],'operation':task['operation'],'value':value,'execution':'deterministic simulated service; not model-generated'}
                    if req.get('trace_id'):self.traces.append(req['trace_id'],'service_result',sim=self.clock,service=event['source'],result=req['response'])
                    self.send(event['source'],req['origin'],'response',req)
                else:self.drop(event['source'],event['request']['origin'],'service failed before completion',event['request'])
                continue
            if not self.links[event['link']]['up'] or not self.agents[event['hop']].up or self.rng.random()<self.config['loss']/100:
                self.drop(event['source'],event['hop'],'link failure, loss or node outage',event.get('request'),event.get('payload'));continue
            self.arrive(event['hop'],event['target'],event['kind'],event['request'],event['visited'],event.get('payload'))
        # One actor per virtual tick; all 32 actors share the same bounded inference queue.
        if self.mode=='network':self.enqueue(list(self.agents)[int(self.clock*2-1)%len(self.agents)])
        else:self.debate.schedule()

    async def loop(self):
        while True:
            if time.time()-self.traces.last_cleanup>=60:self.cleanup()
            if self.running:self.step()
            await asyncio.sleep(.5)

    def cleanup(self):
        now=time.time();self.traces.purge(now)
        self.events=deque((e for e in self.events if e['wall']>now-TTL),maxlen=160)
        for a in self.agents.values():
            a.inbox=deque((e for e in a.inbox if e.get('wall',now)>now-TTL),maxlen=6)
            if a.last_decision.get('_completed_wall',now)<=now-TTL:a.last_decision={}
        for aid,(queued,epoch,obs) in list(self.pending.items()):
            if time.monotonic()-queued>TTL:self.pending.pop(aid,None)
        self.results=deque(x for x in self.results if x[2].get('_completed_wall',now)>now-TTL)
        self.inflight=[x for x in self.inflight if (x[2].get('request') or {}).get('wall',now)>now-TTL] if self.inflight else []
        heapq.heapify(self.inflight)
        if self.mode=='debate' and not self.pending and not self.active and not self.results:self.debate.outstanding=False

    async def launch(self):
        self.workers=[asyncio.create_task(self.worker(i)) for i in range(20)]
        self.loop_task=asyncio.create_task(self.loop())

    async def close(self):
        for task in self.workers+[self.loop_task]:task.cancel()
        await asyncio.gather(*self.workers,self.loop_task,return_exceptions=True)
        await self.client.aclose();self.db.close()

    def snapshot(self):
        cutoff=time.time()-TTL
        self.events=deque((e for e in self.events if e['wall']>cutoff),maxlen=160)
        for a in self.agents.values():
            a.inbox=deque((e for e in a.inbox if e.get('wall',time.time())>cutoff),maxlen=6)
            if a.last_decision.get('_completed_wall',time.time())<=cutoff:a.last_decision={}
        rss=0
        try:
            processes=json.loads((ROOT/'data/processes.json').read_text())
            parent=psutil.Process(processes['model']['pid'])
            rss=sum(p.memory_info().rss for p in [parent]+parent.children(recursive=True) if p.is_running())
        except (OSError,KeyError,ValueError,psutil.Error):pass
        mean=lambda xs:round(sum(xs)/len(xs),3) if xs else 0
        return dict(run=self.run_id,running=self.running,clock=self.clock,mode=self.mode,debate=self.debate.snapshot(),retention_seconds=TTL,model=self.model,health=self.model_health,error=self.model_error,config=self.config,
            agents=[{**{k:v for k,v in asdict(a).items() if k!='inbox'},'inbox':list(a.inbox),'service_queue':round(max(0,a.service_free-self.clock),2)} for a in self.agents.values()],
            links=[{**l,'backlog':round(max([0]+[v-self.clock for v in l['free'].values()]),2)} for l in self.links.values()],
            events=list(self.events)[-80:],pending=list(self.pending),active=list(self.active),buffered=len(self.results),
            metrics=dict(server_rss_mb=round(rss/1024**2),invalid=self.invalid,generated=self.generated,completed=self.completed,dropped=self.dropped,inflight=len(self.inflight),tokens=self.tokens,valid=self.valid,fallbacks=self.failures,network_latency=mean(self.latencies),inference_latency=mean(self.inference_times),queue_latency=mean(self.queue_times)))


Overwriting engine.py


### server.py

In [14]:
%%writefile server.py
import asyncio, json
from contextlib import asynccontextmanager
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from engine import Lab, ROOT

lab=Lab()
@asynccontextmanager
async def lifespan(app):
    await lab.launch()
    yield
    await lab.close()
app=FastAPI(lifespan=lifespan)

@app.middleware('http')
async def local_only(request:Request,call_next):
    # Reject foreign browser origins on state-changing requests.
    if request.method not in ('GET','HEAD'):
        origin=request.headers.get('origin')
        if origin and origin not in ('http://127.0.0.1:8020','http://localhost:8020'):return JSONResponse({'detail':'Local origin required'},status_code=403)
    return await call_next(request)

@app.get('/')
def home():return FileResponse(ROOT/'static/index.html')
@app.get('/api/state')
async def state():return lab.snapshot()
@app.get('/api/health')
async def health():await lab.health();return dict(ready=lab.model_health=='ready',error=lab.model_error)
@app.post('/api/control/{action}')
async def control(action:str):
    if action=='start':
        if not await lab.health():raise HTTPException(503,'Model is not ready. Start it from the notebook or launcher.')
        lab.running=True;lab.log('run','Simulation running with shared vllm-metal inference.')
    elif action=='pause':lab.running=False
    elif action=='step':
        if lab.running:raise HTTPException(409,'Pause before single stepping')
        lab.step()
    elif action=='stop':
        lab.running=False;lab.epoch+=1;lab.pending.clear();lab.results.clear()
        for a in lab.agents.values():a.status='idle'
        lab.log('run','Stopped scheduling. Any in-flight inference result will be discarded; model remains available until launcher stop.')
    elif action=='reset':lab.reset()
    else:raise HTTPException(404,'Unknown command')
    return lab.snapshot()

class Config(BaseModel):
    load:int=Field(1,ge=0,le=5)
    latency:int=Field(100,ge=0,le=3000)
    bandwidth:int=Field(32,ge=1,le=1024)
    loss:int=Field(0,ge=0,le=50)
    service_delay:float=Field(.2,ge=.1,le=10)
    concurrency:int=Field(20,ge=1,le=20)
@app.post('/api/config')
async def config(value:Config):lab.config=value.model_dump();lab.log('config','Traffic and inference settings updated.',config=lab.config);return lab.config
@app.post('/api/toggle/{kind}/{identifier}')
async def toggle(kind:str,identifier:str):
    target=lab.agents.get(identifier) if kind=='agent' else lab.links.get(identifier) if kind=='link' else None
    if target is None:raise HTTPException(404,'Unknown target')
    if kind=='agent':target.up=not target.up
    else:target['up']=not target['up']
    lab.log('fault',f'{identifier} availability toggled.');return lab.snapshot()
@app.get('/api/export')
async def export():
    import time
    rows=lab.db.execute('SELECT payload FROM events WHERE run=? AND wall>? ORDER BY seq',(lab.run_id,time.time()-10800)).fetchall()
    return dict(run=lab.run_id,events=[json.loads(r[0]) for r in rows])

@app.post('/api/mode/{mode}')
async def mode(mode:str):
    if mode not in ('network','debate'):raise HTTPException(400,'Unknown mode')
    lab.mode=mode;lab.reset()
    lab.log('mode','Scientific debate selected: quantum fault tolerance.' if mode=='debate' else 'Network traffic mode selected.')
    return lab.snapshot()

@app.get('/api/conversations/{agent}')
async def conversations(agent:str,direction:str='all'):
    if agent not in lab.agents:raise HTTPException(404,'Unknown agent')
    if direction not in ('all','sent','received'):raise HTTPException(400,'Unknown direction')
    return dict(agent=agent,retention_seconds=10800,items=lab.traces.conversations(agent,direction))

@app.get('/api/traces')
async def traces(kind:str='',agent:str=''):
    return dict(retention_seconds=10800,cleanup_interval_seconds=60,items=lab.traces.recent(kind,agent))

@app.get('/api/traces/{identifier}')
async def trace(identifier:str):
    value=lab.traces.get(identifier)
    if value is None:raise HTTPException(404,'Trace expired or not found')
    return value
@app.websocket('/ws')
async def websocket(ws:WebSocket):
    origin=ws.headers.get('origin')
    if origin not in ('http://127.0.0.1:8020','http://localhost:8020'):await ws.close(code=1008);return
    await ws.accept()
    try:
        while True:await ws.send_json(lab.snapshot());await asyncio.sleep(.5)
    except WebSocketDisconnect:pass
app.mount('/static',StaticFiles(directory=ROOT/'static'),name='static')


Overwriting server.py


### manage.py

In [15]:
%%writefile manage.py
"""Own and stop only the two local services launched by this project."""
import argparse, contextlib, fcntl, json, os, signal, subprocess, sys, time, urllib.request
import psutil
from pathlib import Path
ROOT=Path(__file__).resolve().parent
STATE=ROOT/'data/processes.json'
MODEL='mlx-community/Qwen3-0.6B-4bit'
REVISION='73e3e38d981303bc594367cd910ea6eb48349da8'

def alive(pid):
    try:os.kill(pid,0);return True
    except ProcessLookupError:return False

def status():
    processes=json.loads(STATE.read_text()) if STATE.exists() else {}
    return {k:{**v,'alive':alive(v['pid'])} for k,v in processes.items()}

def probe(port,path):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}{path}',timeout=3) as r:return r.status==200
    except Exception:return False

def start():
    (ROOT/'data').mkdir(exist_ok=True);(ROOT/'logs').mkdir(exist_ok=True)
    with open(ROOT/'data/launcher.lock','w') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX)
        processes=status()
        if not processes.get('model',{}).get('alive'):
            if probe(8011,'/v1/models'):raise RuntimeError('Port 8011 already serves an unowned model; refusing to launch another.')
            import shutil
            exe=shutil.which('vllm')
            if not exe:raise RuntimeError('Install the official vllm-metal Homebrew package first. See README.')
            env={**os.environ,'HF_HOME':str(ROOT/'data/huggingface'),'VLLM_MLX_DEVICE':'gpu','VLLM_METAL_MULTIMODAL_MODE':'text-only'}
            cmd=[exe,'serve',MODEL,'--revision',REVISION,'--host','127.0.0.1','--port','8011','--served-model-name','qwen3-network','--max-model-len','2048','--max-num-seqs','20','--kv-cache-memory-bytes','1073741824','--gpu-memory-utilization','0.15','--generation-config','vllm']
            with open(ROOT/'logs/model.log','ab') as log:
                p=subprocess.Popen(cmd,stdout=log,stderr=log,env=env,cwd=ROOT,start_new_session=True)
            processes['model']={'pid':p.pid,'command':cmd,'started':time.time()}
            STATE.write_text(json.dumps(processes,indent=2))
        if not processes.get('app',{}).get('alive'):
            if probe(8020,'/api/state'):raise RuntimeError('Port 8020 already in use by another app.')
            python=ROOT/'.venv/bin/python'
            cmd=[str(python),'-m','uvicorn','server:app','--host','127.0.0.1','--port','8020']
            with open(ROOT/'logs/app.log','ab') as log:p=subprocess.Popen(cmd,stdout=log,stderr=log,cwd=ROOT,start_new_session=True)
            processes['app']={'pid':p.pid,'command':cmd,'started':time.time()}
            STATE.write_text(json.dumps(processes,indent=2))
    print('Dashboard: http://127.0.0.1:8020\nModel loading/download log: '+str(ROOT/'logs/model.log'))
    return status()

def stop():
    with open(ROOT/'data/launcher.lock','a') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX)
        for name,item in status().items():
            if item['alive']:
                # Check PID identity before stopping the owned process group.
                command=subprocess.check_output(['ps','-p',str(item['pid']),'-o','command='],text=True).strip()
                expected='uvicorn server:app' if name=='app' else 'vllm serve'
                if expected not in command:raise RuntimeError(f'PID {item["pid"]} no longer matches {name}; not stopped.')
                tracked=[]
                with contextlib.suppress(psutil.NoSuchProcess):
                    parent=psutil.Process(item['pid']);tracked=[parent]+parent.children(recursive=True)
                with contextlib.suppress(ProcessLookupError):os.killpg(item['pid'],signal.SIGTERM)
                _,remaining=psutil.wait_procs(tracked,timeout=8)
                for process in remaining:
                    with contextlib.suppress(psutil.NoSuchProcess):process.kill()
                psutil.wait_procs(remaining,timeout=3)
        if STATE.exists():STATE.unlink()
    print('Stopped project-owned app and inference server.')

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('action',choices=['start','stop','status']);a=p.parse_args()
    result={'start':start,'stop':stop,'status':status}[a.action]()
    if result:print(json.dumps(result,indent=2))


Overwriting manage.py


### static/index.html

In [16]:
%%writefile static/index.html
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Agent Network Lab</title><style>
:root{color-scheme:dark;--bg:#050608;--panel:#080a0d;--border:#222934;--text:#f3f7fa;--muted:#a6b0bc;--mint:#9ae6c3;--blue:#4da6ff;--red:#f87171;--accent:#fd8ea1}*{box-sizing:border-box}[hidden]{display:none!important}body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,sans-serif}button,input,select{font:inherit}button{cursor:pointer;background:#0e1218;color:var(--text);border:1px solid var(--border);border-radius:6px;padding:8px 12px}button:hover{background:#181e27}button.primary{background:var(--accent);color:var(--bg);border-color:var(--accent)}button:disabled{opacity:.4;cursor:wait}header{padding:22px 30px;display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--border);gap:15px;flex-wrap:wrap}h1{font-size:20px;letter-spacing:-.03em;margin:0}h1 span{color:var(--accent)}.eyebrow{font:10px ui-monospace,monospace;letter-spacing:.14em;color:var(--muted);text-transform:uppercase}header p{margin:3px 0 0;color:var(--muted);font-size:12px}.actions{display:flex;flex-wrap:wrap;gap:7px}.shell{padding:22px 30px;max-width:1700px;margin:auto}.status{display:flex;gap:15px;align-items:center;flex-wrap:wrap;margin-bottom:18px;color:var(--muted);font-size:12px}.pill{border:1px solid var(--border);padding:5px 10px;border-radius:20px;color:var(--mint)}.metrics{display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin-bottom:20px}.metric{border:1px solid var(--border);border-radius:8px;padding:14px;background:var(--panel)}.metric strong{display:block;font-size:25px;font-weight:500;font-variant-numeric:tabular-nums}.metric span{font-size:11px;color:var(--muted)}.layout{display:grid;grid-template-columns:minmax(0,1fr) 320px;gap:18px}.panel{background:var(--panel);border:1px solid var(--border);border-radius:9px;overflow:hidden}.panel-head{padding:15px 18px;border-bottom:1px solid var(--border);display:flex;justify-content:space-between;gap:12px;align-items:center}.panel-head h2{margin:0;font-size:13px;font-weight:500}.network{padding:5px 15px 15px;background:#090b0f;overflow-x:auto}.network svg{min-width:1100px}svg{width:100%;height:auto;display:block}.link{stroke:#34475e;stroke-width:2;cursor:pointer}.link.off{stroke:var(--red);stroke-dasharray:7 6;opacity:.5}.link.busy{stroke:var(--mint);stroke-width:3}.telemetry{stroke-dasharray:3 6;opacity:.35}.node{cursor:pointer}.node circle{fill:#0e1218;stroke:#4b5666;stroke-width:1.5}.node.selected circle{stroke:var(--mint);stroke-width:3}.node.thinking circle{fill:#27513f;stroke:var(--mint)}.node.off circle{fill:#33222a;stroke:var(--red)}svg text{fill:var(--text);font-size:13px;text-anchor:middle}svg .role{fill:var(--muted);font-size:11px}.legend{padding:0 18px 16px;color:var(--muted);font-size:11px}.dot{fill:var(--mint);pointer-events:none}.inspector{padding:18px}.inspector h3{font-size:23px;margin:0 0 6px}.inspector p{font-size:12px;color:var(--muted)}dl{font-size:12px}dt{color:var(--muted);margin-top:13px}dd{margin:3px 0;overflow-wrap:anywhere}.small{font-size:11px;color:var(--muted)}.queue{display:flex;gap:6px;flex-wrap:wrap;margin-top:12px}.queue span{padding:5px 8px;background:#0e1218;border-radius:4px;font:11px ui-monospace,monospace}.queue .active{background:#254d3e;color:var(--mint)}.lower{display:grid;grid-template-columns:1fr 1.5fr;gap:18px;margin-top:18px}.settings{padding:17px;display:grid;grid-template-columns:1fr 1fr;gap:14px}.settings label{font-size:11px;color:var(--muted)}input,select{width:100%;display:block;margin-top:6px;padding:7px;background:var(--bg);border:1px solid var(--border);color:var(--text);border-radius:4px}.settings button{grid-column:1/-1}.feed{height:310px;overflow:auto;padding:10px 17px}.event{display:grid;grid-template-columns:45px 64px 1fr;gap:10px;padding:9px 0;border-bottom:1px solid #26324477;font-size:11px}.event .kind{color:var(--blue)}.event.fallback .kind,.event.drop .kind{color:var(--red)}.event.decision .kind{color:var(--mint)}.banner{padding:11px 15px;border:1px solid #694552;background:#3c202b55;border-radius:6px;margin-bottom:18px;color:var(--red);font-size:12px}.banner:empty{display:none}footer{padding:25px 0;color:var(--muted);font-size:11px}.replay{display:flex;gap:8px;align-items:center}.replay input{margin:0;width:auto}.modelpath{font:11px ui-monospace,monospace;color:var(--muted);margin-top:14px}@media(max-width:1050px){.metrics{grid-template-columns:repeat(3,1fr)}.layout{grid-template-columns:1fr}.lower{grid-template-columns:1fr}.shell{padding:16px}header{padding:18px}.inspector dl{display:grid;grid-template-columns:1fr 2fr;gap:8px}.inspector dt{margin:0}}@media(max-width:500px){.metrics{grid-template-columns:repeat(2,1fr)}.settings{grid-template-columns:1fr}.event{grid-template-columns:35px 50px 1fr;gap:5px}}
@media(max-width:650px){.network{overflow-x:auto}.network svg{width:1100px;max-width:none}}
@media(prefers-reduced-motion:reduce){.dot{display:none}}
.client-zone{fill:#080a0d;stroke:var(--border)}.zone-title{text-anchor:start!important;fill:var(--accent)!important;font:11px ui-monospace,monospace;letter-spacing:1px}.zone-subtitle{text-anchor:start!important;fill:var(--muted)!important;font-size:11px}.diagram-label{fill:var(--muted)!important;font:11px ui-monospace,monospace;letter-spacing:1px}.bundle{stroke:#526477;stroke-width:3}.bundle.degraded{stroke:var(--red)}.bundle-label{font-size:10px!important;fill:var(--muted)!important}.node rect{fill:#0e1218;stroke:#4b5666;stroke-width:1.3}.node.selected rect{stroke:var(--accent);stroke-width:2.5}.node.thinking rect{fill:#1b352b;stroke:var(--mint)}.node.off rect{fill:#33222a;stroke:var(--red)}.client-name{font-size:11px!important}.client-state{font-size:10px!important;fill:var(--muted)!important}.infra-name{font-size:12px!important}.model-box{fill:#171017;stroke:var(--accent)}.model-title{font-size:12px!important;fill:var(--accent)!important}.inference-bus{stroke:var(--accent);stroke-width:1.5;stroke-dasharray:5 5;opacity:.7}.telemetry{opacity:.22}.link{stroke-width:2.5}.network svg{min-width:1150px}
.mode-controls{padding:16px 18px;margin-bottom:18px;display:flex;gap:24px;align-items:center;justify-content:space-between}.mode-controls p{max-width:520px}.debate-intro{padding:18px}.debate-intro h3{font-size:22px;font-weight:500;margin:0 0 12px}.debate-intro p{font-size:13px;color:var(--muted)}.debate-intro a{color:var(--blue)}.debate-intro details{font-size:12px}.debate-transcript{max-height:420px;overflow:auto;padding:0 18px 18px}.argument{padding:18px 0;border-top:1px solid var(--border)}.argument p{font-size:14px}.argument .eyebrow{color:var(--accent)}#debate-panel{margin-bottom:18px}.trace-panel{margin-top:18px}.trace-tools{display:flex;gap:18px;align-items:center;padding:16px;flex-wrap:wrap}.trace-tools label{font-size:12px}.trace-tools select{min-width:180px}.trace-tools span{max-width:600px}.trace-layout{display:grid;grid-template-columns:320px minmax(0,1fr)}.trace-list{max-height:480px;overflow:auto;padding:12px}.trace-row{display:block;width:100%;text-align:left;margin-bottom:7px;padding:12px}.trace-row strong{display:block;font-size:12px;font-weight:500}.trace-row span{display:block;font-size:10px;color:var(--muted);margin-top:5px}.trace-detail{padding:18px;border-left:1px solid var(--border)}.trace-detail h3{margin:0 0 10px;font-size:18px}.trace-detail pre{max-height:360px;overflow:auto;white-space:pre-wrap;overflow-wrap:anywhere;background:var(--bg);padding:14px;font:11px/1.6 ui-monospace,monospace;border:1px solid var(--border)}@media(max-width:800px){.trace-layout{grid-template-columns:1fr}.trace-list{max-height:240px}.trace-detail{border-left:0;border-top:1px solid var(--border)}.mode-controls{display:block}}
.conversations-panel{margin-top:18px}.conversation-controls{display:flex;gap:16px;align-items:end;padding:16px;flex-wrap:wrap}.conversation-controls label{font-size:12px}.conversation-controls select{min-width:190px}.conversation-note{padding:0 18px}.conversation-messages{padding:0 18px 18px;max-height:520px;overflow:auto}.conversation-bubble{padding:16px;margin:12px 0;border:1px solid var(--border);border-radius:9px;max-width:92%;background:#0e1218}.conversation-bubble.outgoing{margin-left:auto;border-left:3px solid var(--accent)}.conversation-bubble.incoming{margin-right:auto;border-left:3px solid var(--blue)}.conversation-bubble .eyebrow{color:var(--text);letter-spacing:.03em}.conversation-bubble p{white-space:pre-wrap;overflow-wrap:anywhere}.conversation-bubble .small{font-size:11px}.conversation-bubble button{font-size:11px}@media(max-width:600px){.conversation-bubble{max-width:100%}.conversation-controls select{min-width:150px}}
</style></head><body>
<header><div><div class="eyebrow">Local multi-agent systems / Metal</div><h1>Agent <span>Network Lab</span></h1><p>25 clients · 7 network agents · One shared model.</p></div><div class="actions"><button class="primary" data-command="start">Start</button><button data-command="pause">Pause</button><button data-command="step">Step</button><button data-command="stop">Stop run</button><button data-command="reset">Reset</button></div></header>
<main class="shell"><div class="status"><span class="pill" id="health">Connecting…</span><span id="run-state">Paused</span><span id="clock">0.0 simulated seconds</span><span>Qwen3-0.6B · MLX 4-bit · vllm-metal</span><button id="check">Check model</button></div><div id="error" class="banner" role="alert"></div>
<div class="metrics"><div class="metric"><strong id="completed">0</strong><span>Completed requests</span></div><div class="metric"><strong id="inflight">0</strong><span>Network events pending</span></div><div class="metric"><strong id="dropped">0</strong><span>Dropped messages</span></div><div class="metric"><strong id="network_latency">0</strong><span>Mean RTT · virtual seconds</span></div><div class="metric"><strong id="inference_latency">0</strong><span>Inference · real seconds</span></div><div class="metric"><strong id="valid">0</strong><span>Validated model decisions</span></div></div>
<section class="mode-controls panel"><div><span class="eyebrow">Choose the experiment</span><div class="actions" style="margin-top:8px"><button id="network-mode">Network traffic</button><button id="debate-mode">Scientific debate</button></div></div><p class="small">One shared Qwen model in both modes. Switching modes starts a fresh run; existing traces expire after 3 hours.</p></section>
<section id="debate-panel" class="panel" hidden>
 <div class="panel-head"><h2>Scientific debate · Quantum fault tolerance</h2><span id="debate-progress" class="pill">0 / 100 turns</span></div>
 <div class="debate-intro"><h3 id="debate-topic"></h3><p>25 researchers · 5 teams · Opening claims → Cross-examination → Rebuttal → Synthesis. Arguments are model-generated; evidence IDs indicate a supplied source, not verified factual correctness. All participants share the same tiny model, so agreement is not independent scientific confirmation.</p><p class="small">The shared transcript is supplied centrally; argument delivery is also visualized through the simulated network. No winner is automatically declared.</p><details><summary>Read the evidence cards</summary><div id="evidence-cards"></div></details></div>
 <div id="debate-transcript" class="debate-transcript" aria-label="Scientific debate transcript"></div>
</section>

<div class="layout"><section class="panel"><div class="panel-head"><h2>Network topology</h2><span class="small">Select a client to inspect its link · Click infrastructure links to fail / restore</span></div><div class="network"><svg id="topology" viewBox="0 0 1150 730" role="group" aria-label="25 clients connected through gateways, routers and services"></svg></div><div class="legend">Client links are bundled by gateway; select a client to inspect its individual connection. Dashed gray: observer telemetry. Pink bus: shared inference (logical connection, not simulated traffic).</div></section>
<aside class="panel"><div class="panel-head"><h2>Agent inspector</h2><button id="toggle-agent">Fail / restore</button></div><div class="inspector"><h3 id="agent-name">Client 01</h3><div class="eyebrow" id="agent-role">Client</div><dl id="agent-info"></dl><p id="decision">Waiting for a decision.</p><hr style="border:0;border-top:1px solid var(--border)"><div class="eyebrow">Shared inference queue</div><div class="queue" id="queue"></div><p id="queue-info"></p><div class="modelpath">32 actors → shared inference queue → one model</div><p class="small" id="tokens"></p></div></aside></div>
<div class="lower"><section class="panel"><div class="panel-head"><h2>Traffic & inference controls</h2></div><form id="config" class="settings"><label>Requests per client / 3 virtual sec<input name="load" type="number" value="1" min="0" max="5"></label><label>Link latency (ms, simulated)<input name="latency" type="number" value="100" min="0" max="3000"></label><label>Bandwidth (KiB/s, simulated)<input name="bandwidth" type="number" value="32" min="1" max="1024"></label><label>Loss probability (%)<input name="loss" type="number" value="0" min="0" max="50"></label><label>Service time (virtual sec)<input name="service_delay" type="number" value="0.2" min="0.1" max="10" step="0.1"></label><label>Shared inference concurrency<select name="concurrency"><option>1</option><option>2</option></select></label><button type="submit">Apply settings</button></form></section>
<section class="panel"><div class="panel-head"><h2 id="feed-title">Live event stream</h2><div class="replay"><button id="export">Export / playback</button><button id="live" hidden>Return live</button></div></div><input id="scrub" type="range" min="0" value="0" hidden aria-label="Recorded event position"><div class="feed" id="feed" aria-label="Network and agent event log"></div></section></div>
<section class="panel conversations-panel">
 <div class="panel-head"><h2>Agent conversations</h2><span class="small">Actual generated messages · 3-hour retention</span></div>
 <div class="conversation-controls"><label>Choose an agent<select id="conversation-agent"></select></label><label>Messages<select id="conversation-direction"><option value="all">Sent & incoming</option><option value="sent">Sent by this agent</option><option value="received">Delivered to this agent</option></select></label><button id="conversation-refresh">Refresh messages</button></div>
 <p class="small conversation-note">Select a node in the diagram or choose an agent above. Each bubble shows its sender, recipient and delivery state. “Generated only” means an older model output whose network delivery was not recorded. Status messages are network observations; service requests and responses are deterministic simulation tasks; scientific arguments are unverified model output.</p>
 <div id="conversation-messages" class="conversation-messages" aria-label="Selected agent conversation"></div>
</section>

<section class="panel trace-panel">
 <div class="panel-head"><h2>What are they serving? · Full traces</h2><span class="small">3-hour wall-clock retention</span></div>
 <div class="trace-tools"><label>Trace type<select id="trace-kind"><option value="">All types</option><option value="request">Service requests</option><option value="inference">Network model decisions</option><option value="debate">Scientific arguments</option></select></label><button id="refresh-traces">Refresh traces</button><span class="small">Expired traces are inaccessible immediately; disk cleanup runs every 60 seconds while the app is running and on restart. Downloaded exports are outside retention.</span></div>
 <div class="trace-layout"><div id="trace-list" class="trace-list" aria-label="Recent traces"></div><div class="trace-detail"><h3 id="trace-title">Select a trace</h3><p id="trace-expiry" class="small"></p><p class="small">Service requests show deterministic sample tasks and results. Inference traces show the actual messages and model response; debate traces show source cards, arguments and replies. Private internal model reasoning is not captured.</p><pre id="trace-json">No trace selected.</pre></div></div>
</section>

<footer>Simulation only: 0.5-second virtual ticks; network controls do not change macOS networking. Green decisions are model-generated; red fallbacks are deterministic and explicit. Stop run stops scheduling; use the notebook’s shutdown cell to unload the model and stop both services. <span id="connection"></span></footer></main>
<script>
const $=id=>document.getElementById(id),NS='http://www.w3.org/2000/svg';let state=null,selected='C01',recording=null,configDirty=false;
const positions={A4:[532,208],A5:[532,493],A6:[760,208],A7:[760,493],A8:[1010,208],A9:[1010,493],A10:[760,48]};
for(let i=0;i<25;i++){const n=i<13?i:i-13;positions['C'+String(i+1).padStart(2,'0')]=[62+(n%5)*72,(i<13?163:448)+Math.floor(n/5)*60];}
function agentLabel(a){if(state?.mode==='debate'&&a.role==='Client')return 'Researcher '+a.id.slice(1);return a.role==='Client'?'Client '+a.id.slice(1):a.role+' '+({A4:1,A5:2,A6:1,A7:2,A8:1,A9:2,A10:1}[a.id]);}
function el(tag,attrs,text){const e=document.createElementNS(NS,tag);for(const[k,v]of Object.entries(attrs))e.setAttribute(k,v);if(text!==undefined)e.textContent=text;return e}
async function request(path,body={}){try{const r=await fetch(path,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(body)});const data=await r.json();if(!r.ok)throw Error(data.detail||r.statusText);$('error').textContent='';return data}catch(e){$('error').textContent=e.message;return null}}
function graph(s){
 if(s.mode==='debate'){graphDebate(s);return}
 const svg=$('topology');svg.replaceChildren();
 const text=(x,y,value,cls='diagram-label')=>svg.append(el('text',{x,y,class:cls},value));
 const path=(d,cls,attrs={})=>{const p=el('path',{d,class:cls,fill:'none',...attrs});svg.append(p);return p};
 const nodeStates=Object.fromEntries(s.agents.map(a=>[a.id,a]));
 for(const [y,title,sub] of [[80,'CLIENT GROUP 1','13 clients · C01–C13 → Gateway 1'],[365,'CLIENT GROUP 2','12 clients · C14–C25 → Gateway 2']]){
  svg.append(el('rect',{x:18,y,width:390,height:245,rx:10,class:'client-zone'}));
  text(38,y+25,title,'zone-title');text(38,y+46,sub,'zone-subtitle');
 }
 text(532,110,'GATEWAYS');text(760,110,'ROUTERS');text(1010,110,'SERVICES');
 // Bundled client edges reduce clutter. The selected client's physical link remains actionable.
 for(const [gateway,y,label] of [['A4',208,'13 client links'],['A5',493,'12 client links']]){
  const links=s.links.filter(l=>l.b===gateway&&l.a.startsWith('C'));
  const broken=links.filter(l=>!l.up).length;
  path(`M408 ${y} H478`,'bundle'+(broken?' degraded':''));
  text(445,y-20,broken?`${broken} / ${links.length} links down`:label,'bundle-label');
 }
 for(const link of s.links){
  const isClient=link.a.startsWith('C');
  if(isClient&&link.a!==selected)continue;
  const observer=link.a==='A10';
  const a=positions[link.a],b=positions[link.b];
  let d;
  if(isClient)d=`M408 ${link.b==='A4'?208:493} H478`;
  else if(observer)d=`M${a[0]} ${a[1]+24} V135 H${b[0]} V${b[1]-29}`;
  else {const right=a[0]<b[0];const ax=a[0]+(right?54:-54),bx=b[0]+(right?-54:54);const middle=(ax+bx)/2;d=`M${ax} ${a[1]} C${middle} ${a[1]} ${middle} ${b[1]} ${bx} ${b[1]}`;}
  const line=path(d,'link '+(!link.up?'off':link.backlog>0?'busy':'')+(observer?' telemetry':''),{role:'button',tabindex:0,'aria-label':`${link.id}: ${link.up?'healthy':'failed'}. Toggle link.`});
  const toggle=()=>request('/api/toggle/link/'+link.id);line.onclick=toggle;line.onkeydown=e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();toggle()}};
  line.append(el('title',{},`${link.id} · click to ${link.up?'fail':'restore'}${isClient?' (selected client)':''}`));
  if(link.up&&s.running&&link.backlog>0&&!observer){const dot=el('circle',{r:3,class:'dot'});dot.append(el('animateMotion',{dur:'1.3s',repeatCount:'indefinite',path:d}));svg.append(dot)}
 }
 for(const a of s.agents){
  const[x,y]=positions[a.id],client=a.role==='Client';
  const group=el('g',{class:'node '+(a.id===selected?'selected ':'')+(a.up?a.status:'off'),transform:`translate(${x},${y})`,role:'button',tabindex:0,'aria-label':`${agentLabel(a)} (${a.id}), ${a.up?a.status:'offline'}`});
  group.append(el('rect',{x:client?-30:-54,y:client?-22:-29,width:client?60:108,height:client?44:58,rx:client?7:10}));
  if(client){group.append(el('text',{y:-3,class:'client-name'},'Client '+a.id.slice(1)),el('text',{y:12,class:'client-state'},a.up?a.status:'offline'))}
  else {group.append(el('text',{y:-3,class:'infra-name'},agentLabel(a)),el('text',{y:16,class:'client-state'},a.id+' · '+(a.up?a.status:'offline')))}
  group.append(el('title',{},`${agentLabel(a)} · select to inspect messages, model decisions, and availability`));
  const choose=()=>{selected=a.id;render(state)};group.onclick=choose;group.onkeydown=e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();choose()}};
  svg.append(group);
 }
 text(760,599,'ALL 32 ACTORS SHARE INFERENCE','diagram-label');
 path('M532 530 V630 H1010 V530','inference-bus');
 path('M760 630 V652','inference-bus');
 svg.append(el('rect',{x:575,y:652,width:370,height:52,rx:10,class:'model-box'}));
 text(760,674,'ONE QWEN3-0.6B MODEL','model-title');
 text(760,692,'Shared queue → vllm-metal → Metal GPU','client-state');
}
function feed(events){const box=$('feed');box.replaceChildren();for(const e of events.slice(-60).reverse()){const row=document.createElement('div');row.className='event '+e.kind;for(const [className,text] of [['small',Number(e.sim).toFixed(1)],['kind',e.kind],['',e.text]]){const c=document.createElement('span');c.className=className;c.textContent=text;row.append(c)}box.append(row)}}
function render(s){if(!s)return;state=s;if(s.mode==='debate'&&!selected.startsWith('C'))selected='C01';renderLearning(s);syncConversations(s);if(!configDirty){for(const [key,value]of Object.entries(s.config)){const field=$('config').elements.namedItem(key);if(field)field.value=value}}$('health').textContent=s.health==='ready'?'● Model ready':'○ Model '+s.health;$('run-state').textContent=s.running?'RUNNING':'PAUSED';$('clock').textContent=s.clock.toFixed(1)+' simulated seconds';for(const k of ['completed','inflight','dropped','network_latency','inference_latency','valid'])$(k).textContent=s.metrics[k];if(s.mode==='debate'){$('completed').textContent=s.debate.turn;$('completed').nextElementSibling.textContent='Published arguments';$('network_latency').textContent=s.debate.entries.filter(e=>e.reply_to).length;$('network_latency').nextElementSibling.textContent='Replies to an earlier argument'}else{$('completed').nextElementSibling.textContent='Completed requests';$('network_latency').nextElementSibling.textContent='Mean RTT · virtual seconds'}if(s.error)$('error').textContent=s.error;else if(s.health==='ready')$('error').textContent='';graph(s);const a=s.agents.find(a=>a.id===selected);$('agent-name').textContent=agentLabel(a)+' · '+a.id;$('agent-role').textContent=(s.mode==='debate'&&a.role==='Client'?s.debate.teams[Math.floor((Number(a.id.slice(1))-1)/5)].name:a.role)+' · '+(a.up?a.status:'OFFLINE');$('agent-info').replaceChildren();for(const[k,v]of [['Messages sent / received',a.sent+' / '+a.received],['Decisions',a.decisions],['Service preference',a.preferred_service],['Preferred next hop',a.preferred_hop||'Shortest valid path'],['Service queue (virtual sec)',a.service_queue]]){const dt=document.createElement('dt'),dd=document.createElement('dd');dt.textContent=k;dd.textContent=v;$('agent-info').append(dt,dd)}$('decision').textContent=a.last_decision.action?`${a.last_decision.fallback?'FALLBACK':'MODEL'}: ${a.last_decision.action} — ${a.last_decision.reason}`:'Waiting for a decision.';$('queue').replaceChildren();for(const id of [...s.active,...s.pending]){const badge=document.createElement('span');badge.textContent=id;badge.className=s.active.includes(id)?'active':'';$('queue').append(badge)}$('queue-info').textContent=`${s.active.length} active · ${s.pending.length} waiting · ${s.buffered} buffered. Mean queue: ${s.metrics.queue_latency}s`;$('tokens').textContent=`${s.metrics.tokens} inference tokens · ${s.metrics.fallbacks} fallbacks · ${s.metrics.invalid} invalid outputs. Model process-tree RSS: ${s.metrics.server_rss_mb} MiB (shared pages may be counted more than once). GPU utilization unavailable.`;if(!recording)feed(s.events)}
document.querySelectorAll('[data-command]').forEach(b=>b.onclick=async()=>{const s=await request('/api/control/'+b.dataset.command);if(s)render(s)});$('toggle-agent').onclick=()=>request('/api/toggle/agent/'+selected);$('config').oninput=()=>{configDirty=true};$('config').onsubmit=async e=>{e.preventDefault();const values=Object.fromEntries([...new FormData(e.target)].map(([k,v])=>[k,Number(v)]));const saved=await request('/api/config',values);if(saved)configDirty=false};$('check').onclick=async()=>{const r=await fetch('/api/health');const d=await r.json();$('error').textContent=d.ready?'':d.error};
$('export').onclick=async()=>{try{const r=await fetch('/api/export');recording=await r.json();$('scrub').hidden=false;$('scrub').max=Math.max(0,recording.events.length-1);$('scrub').value=$('scrub').max;$('live').hidden=false;$('feed-title').textContent='Recorded event playback (topology remains live)';feed(recording.events);const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([JSON.stringify(recording,null,2)],{type:'application/json'}));a.download='network-run-'+recording.run+'.json';a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000)}catch(e){$('error').textContent=e.message}};$('scrub').oninput=()=>feed(recording.events.slice(0,Number($('scrub').value)+1));$('live').onclick=()=>{recording=null;$('scrub').hidden=true;$('live').hidden=true;$('feed-title').textContent='Live event stream';render(state)};
function connect(){const ws=new WebSocket('ws://'+location.host+'/ws');ws.onmessage=e=>{render(JSON.parse(e.data));$('connection').textContent='Live connection active.'};ws.onclose=()=>{$('connection').textContent='Disconnected. Reconnecting…';setTimeout(connect,2000)}}connect();fetch('/api/state').then(r=>r.json()).then(render);fetch('/api/health').catch(()=>{});setInterval(()=>{fetch('/api/health').catch(()=>{})},5000);
let selectedTrace=null,lastTraceRefresh=0,debateRenderKey='';
async function loadTrace(id){try{const r=await fetch('/api/traces/'+encodeURIComponent(id));if(!r.ok)throw Error('Trace expired or unavailable');selectedTrace=await r.json();$('trace-title').textContent=selectedTrace.payload.title||selectedTrace.kind;$('trace-json').textContent=JSON.stringify(selectedTrace,null,2);updateExpiry()}catch(e){selectedTrace=null;$('trace-title').textContent='Trace unavailable';$('trace-json').textContent=e.message;$('trace-expiry').textContent=''}}
function updateExpiry(){if(!selectedTrace)return;const remaining=selectedTrace.expires-Date.now()/1000;if(remaining<=0){selectedTrace=null;$('trace-title').textContent='Trace expired';$('trace-json').textContent='This trace has been deleted by the retention policy.';$('trace-expiry').textContent='';return}$('trace-expiry').textContent='Auto-deletes in '+Math.floor(remaining/3600)+'h '+Math.floor(remaining%3600/60)+'m '+Math.floor(remaining%60)+'s · '+new Date(selectedTrace.expires*1000).toLocaleTimeString()}
async function refreshTraces(){try{const r=await fetch('/api/traces?kind='+encodeURIComponent($('trace-kind').value));if(!r.ok)return;const data=await r.json();$('trace-list').replaceChildren();for(const t of data.items){const b=document.createElement('button');b.className='trace-row';const title=document.createElement('strong'),info=document.createElement('span');title.textContent=t.title;info.textContent=`${t.agent} · ${t.kind} · ${t.status} · ${new Date(t.created*1000).toLocaleTimeString()}`;b.append(title,info);b.onclick=()=>loadTrace(t.id);$('trace-list').append(b)}if(!data.items.length)$('trace-list').textContent='No retained traces yet. Start a run to record activity.'}catch{}}
function renderLearning(s){document.querySelector('.layout .panel-head h2').textContent=s.mode==='debate'?'Research teams & shared inference':'Network topology';$('topology').setAttribute('aria-label',s.mode==='debate'?'25 scientific researchers in five teams sharing one model':'25 clients connected through gateways, routers and services');document.querySelector('.legend').textContent=s.mode==='debate'?'Five teams of five researchers. Colored curves show recent replies. Pink paths represent the single shared inference resource. All statements are unverified model output.':'Client links are bundled by gateway; select a client to inspect its individual connection. Dashed gray: observer telemetry. Pink bus: shared inference, not simulated traffic.';$('network-mode').classList.toggle('primary',s.mode==='network');$('debate-mode').classList.toggle('primary',s.mode==='debate');$('debate-panel').hidden=s.mode!=='debate';$('debate-topic').textContent=s.debate.topic;$('debate-progress').textContent=s.debate.phase+' · '+s.debate.turn+' / '+s.debate.total_turns;const key=s.run+':'+s.debate.turn+':'+s.debate.entries.length;if(key!==debateRenderKey){debateRenderKey=key;$('debate-transcript').replaceChildren();for(const e of s.debate.entries.slice().reverse()){const article=document.createElement('article');article.className='argument';const heading=document.createElement('div');heading.className='eyebrow';heading.textContent=`${e.id} · ${e.agent} · ${e.team} · ${e.phase}${e.reply_to?' · replies to '+e.reply_to:''}`;const p=document.createElement('p');p.textContent=e.argument;const caveat=document.createElement('p');caveat.className='small';caveat.textContent=`${e.fallback?'FALLBACK · ':''}Source ${e.evidence_id||'none'} · Caveat: ${e.uncertainty}`;const trace=document.createElement('button');trace.textContent='Inspect prompt & response';trace.onclick=()=>{loadTrace(e.trace_id);$('trace-title').scrollIntoView({behavior:'smooth',block:'center'})};article.append(heading,p,caveat,trace);$('debate-transcript').append(article)}if(!s.debate.entries.length)$('debate-transcript').textContent='Select Scientific debate, then Start. Each participant will receive a role, evidence cards and an earlier argument to address.';$('evidence-cards').replaceChildren();for(const c of s.debate.evidence){const p=document.createElement('p'),a=document.createElement('a');a.href=c.url;a.target='_blank';a.rel='noopener';a.textContent=c.id+' · '+c.title;p.append(a,document.createElement('br'),document.createTextNode(c.fact));$('evidence-cards').append(p)}}if(Date.now()-lastTraceRefresh>5000){lastTraceRefresh=Date.now();refreshTraces()}updateExpiry()}
function graphDebate(s){const svg=$('topology');svg.replaceChildren();const colors=['#9ae6c3','#89b8ff','#c3a6ff','#efbe82','#fd8ea1'];const dp={};for(let team=0;team<5;team++){const x=18+team*224;svg.append(el('rect',{x,y:90,width:210,height:420,rx:10,class:'client-zone'}));svg.append(el('text',{x:x+105,y:120,class:'diagram-label'},s.debate.teams[team].name));for(let j=0;j<5;j++){const aid='C'+String(team*5+j+1).padStart(2,'0');dp[aid]=[x+105,165+j*72]}const line=el('path',{d:`M${x+105} 510 V580 H575 V652`,class:'inference-bus',fill:'none'});svg.append(line)}for(const entry of s.debate.entries.slice(-5)){const from=dp[entry.agent],to=dp[entry.recipient];if(from&&to)svg.append(el('path',{d:`M${from[0]} ${from[1]} Q575 35 ${to[0]} ${to[1]}`,fill:'none',stroke:colors[Math.floor((Number(entry.agent.slice(1))-1)/5)],'stroke-width':1.5,opacity:.45}))}for(const a of s.agents.filter(a=>a.role==='Client')){const[x,y]=dp[a.id],n=Number(a.id.slice(1));const g=el('g',{class:'node '+(a.id===selected?'selected ':'')+(a.up?a.status:'off'),transform:`translate(${x},${y})`,role:'button',tabindex:0,'aria-label':`Researcher ${n}, ${s.debate.teams[Math.floor((n-1)/5)].name}, ${a.status}`});g.append(el('rect',{x:-76,y:-25,width:152,height:50,rx:8}),el('text',{y:-3,class:'infra-name'},'Researcher '+String(n).padStart(2,'0')),el('text',{y:16,class:'client-state'},a.id+' · '+a.status));g.onclick=()=>{selected=a.id;render(state)};g.onkeydown=e=>{if(e.key==='Enter')g.onclick()};svg.append(g)}svg.append(el('rect',{x:385,y:652,width:380,height:52,rx:10,class:'model-box'}),el('text',{x:575,y:675,class:'model-title'},'ONE SHARED QWEN3-0.6B'),el('text',{x:575,y:694,class:'client-state'},'25 roles · one model, not independent experts'));}
$('network-mode').onclick=async()=>{const s=await request('/api/mode/network');if(s)render(s)};$('debate-mode').onclick=async()=>{const s=await request('/api/mode/debate');if(s)render(s)};$('refresh-traces').onclick=refreshTraces;$('trace-kind').onchange=refreshTraces;setInterval(()=>{updateExpiry();if(state){const retained=state.debate.entries.filter(e=>Date.now()/1000-e.wall<10800);if(retained.length!==state.debate.entries.length){state.debate.entries=retained;debateRenderKey='';renderLearning(state)}}},1000);

let conversationCache=[],conversationAgent='',conversationUpdated=0,conversationSelection='';
function displayAgent(id){const a=state?.agents.find(x=>x.id===id);return a?agentLabel(a)+' ('+id+')':id}
function renderConversation(){const container=$('conversation-messages');container.replaceChildren();const retained=conversationCache.filter(x=>x.expires>Date.now()/1000);for(const item of retained){const card=document.createElement('article');card.className='conversation-bubble '+(item.source===selected?'outgoing':'incoming');const header=document.createElement('div');header.className='eyebrow';header.textContent=displayAgent(item.source)+' → '+displayAgent(item.recipient);const meta=document.createElement('p');meta.className='small';meta.textContent=`${item.source===selected?'SENT':'INCOMING'} · ${({debate:'Scientific argument',status:'Network status',request:'Simulated service request',response:'Simulated service response'}[item.kind]||item.kind)} · ${item.status.replaceAll('_',' ')} · ${new Date(item.created*1000).toLocaleTimeString()}`;const body=document.createElement('p');body.textContent=item.text;card.append(header,meta,body);if(item.reply_to||item.evidence_id||item.uncertainty){const note=document.createElement('p');note.className='small';note.textContent=[item.reply_to?'Replies to '+item.reply_to:'',item.evidence_id?'Source '+item.evidence_id:'',item.uncertainty?'Caveat: '+item.uncertainty:''].filter(Boolean).join(' · ');card.append(note)}const button=document.createElement('button');button.textContent='Inspect full trace';button.onclick=()=>{loadTrace(item.trace_id);$('trace-title').scrollIntoView({behavior:'smooth',block:'center'})};card.append(button);container.append(card)}if(!retained.length)container.textContent='No matching retained messages for this agent yet. Start or resume the simulation, or choose Sent & incoming to see generated and in-transit messages.'}
async function refreshConversation(){const aid=$('conversation-agent').value||selected,direction=$('conversation-direction').value,key=aid+':'+direction;conversationSelection=key;try{const response=await fetch('/api/conversations/'+encodeURIComponent(aid)+'?direction='+direction);if(!response.ok)return;const data=await response.json();if(conversationSelection!==key)return;conversationCache=data.items;conversationAgent=aid;conversationUpdated=Date.now();renderConversation()}catch{}}
function syncConversations(s){const picker=$('conversation-agent');if(picker.options.length!==s.agents.length){picker.replaceChildren();for(const a of s.agents){const option=document.createElement('option');option.value=a.id;option.textContent=displayAgent(a.id);picker.append(option)}}else{for(const option of picker.options)option.textContent=displayAgent(option.value)}picker.value=selected;if(conversationAgent!==selected||Date.now()-conversationUpdated>3000){conversationUpdated=Date.now();refreshConversation()}}
$('conversation-agent').onchange=()=>{selected=$('conversation-agent').value;conversationCache=[];renderConversation();refreshConversation();if(state)render(state)};$('conversation-direction').onchange=refreshConversation;$('conversation-refresh').onclick=refreshConversation;setInterval(()=>{if(conversationCache.some(x=>x.expires<=Date.now()/1000)){conversationCache=conversationCache.filter(x=>x.expires>Date.now()/1000);renderConversation()}},1000);

</script></body></html>


Overwriting static/index.html


### test_engine.py

In [17]:
%%writefile test_engine.py
import asyncio, json
import httpx, pytest
from engine import Lab

@pytest.fixture
def lab(tmp_path):
    l=Lab(tmp_path/'test.sqlite')
    yield l
    l.db.close()

def test_delivery_roundtrip(lab):
    for _ in range(100):lab.step()
    assert lab.completed>0
    assert lab.completed<=lab.generated
    assert len(lab.agents)==32
    assert sum(a.role=='Client' for a in lab.agents.values())==25

def test_service_outage_uses_alternative(lab):
    lab.agents['A8'].up=False
    for _ in range(100):lab.step()
    assert lab.agents['A9'].received>0
    assert lab.completed>0

def test_router_outage_cannot_route_through_observer(lab):
    lab.agents['A6'].up=lab.agents['A7'].up=False
    assert lab.next_hop('A4','A8',[]) is None

def test_bounded_inference_and_stale_results(lab):
    for _ in range(500):lab.enqueue('C01')
    assert list(lab.pending)==['C01']
    epoch=lab.epoch;lab.reset()
    lab.results.append((epoch,'C01',{'action':'report_status','reason':'old'},None,0,0))
    lab.apply_results()
    assert lab.agents['C01'].decisions==0

def test_full_queue_drops(lab):
    lab.config['bandwidth']=1
    for _ in range(50):lab.send('C01','A8','request',{'id':'x','origin':'C01','created':0})
    assert lab.dropped>0

@pytest.mark.asyncio
async def test_invalid_action_never_executes(lab):
    calls=[]
    def reply(request):
        calls.append(request)
        return httpx.Response(200,json={'choices':[{'message':{'content':json.dumps({'action':'run_shell','target':'','reason':'bad'})}}]})
    await lab.client.aclose();lab.client=httpx.AsyncClient(transport=httpx.MockTransport(reply))
    decision,error=await lab.infer('C01',lab.observation('C01'))
    assert len(calls)==2 and error and decision['action']=='report_status'
    await lab.client.aclose()

@pytest.mark.asyncio
async def test_model_output_validated(lab):
    def reply(request):return httpx.Response(200,json={'choices':[{'message':{'content':'{"action":"set_service","target":"A9","reason":"Less load"}'}}]})
    await lab.client.aclose();lab.client=httpx.AsyncClient(transport=httpx.MockTransport(reply))
    decision,error=await lab.infer('A4',lab.observation('A4'))
    assert error is None and decision['target']=='A9'
    await lab.client.aclose()

@pytest.mark.asyncio
async def test_offline_service_rejected(lab):
    lab.agents['A8'].up=False
    def reply(request):return httpx.Response(200,json={'choices':[{'message':{'content':'{"action":"set_service","target":"A8","reason":"Wrong"}'}}]})
    await lab.client.aclose();lab.client=httpx.AsyncClient(transport=httpx.MockTransport(reply))
    decision,error=await lab.infer('A4',lab.observation('A4'))
    assert error and decision['action']=='report_status'
    await lab.client.aclose()

@pytest.mark.asyncio
async def test_pause_buffers_model_result(lab):
    lab.results.append((lab.epoch,'A4',{'action':'set_service','target':'A9','reason':'Test'},None,0,.1))
    await lab.launch()
    await asyncio.sleep(.6)
    assert lab.agents['A4'].preferred_service=='A8'
    lab.step()
    assert lab.agents['A4'].preferred_service=='A9'
    # Close tasks and HTTP client here; fixture closes the database.
    for task in lab.workers+[lab.loop_task]:task.cancel()
    await asyncio.gather(*lab.workers,lab.loop_task,return_exceptions=True)
    await lab.client.aclose()

@pytest.mark.asyncio
async def test_control_endpoints_write_events(lab,monkeypatch):
    import server
    monkeypatch.setattr(server,'lab',lab)
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=server.app),base_url='http://localhost:8020') as c:
        r=await c.post('/api/config',json={**lab.config,'concurrency':2})
        assert r.status_code==200
        r=await c.post('/api/toggle/agent/A8')
        assert r.status_code==200 and not lab.agents['A8'].up
        r=await c.get('/api/export')
        assert len(r.json()['events'])>=3
        r=await c.post('/api/config',json={**lab.config,'concurrency':10})
        assert r.status_code==422
        r=await c.post('/api/control/reset',headers={'Origin':'https://example.com'})
        assert r.status_code==403
    await lab.client.aclose()


Overwriting test_engine.py


## Tests
These deterministic tests include mocked inference for validation failure cases. Real-server results are recorded separately in VERIFICATION.json.

In [18]:
subprocess.run([str(ROOT/".venv/bin/python"), "-m", "pytest", "-q"], check=True)

.........F.F......                                                       [100%]
=================================== FAILURES ===================================
_____________________ test_control_endpoints_write_events ______________________

lab = <engine.Lab object at 0x10bce68d0>
monkeypatch = <_pytest.monkeypatch.MonkeyPatch object at 0x10bce4050>

    @pytest.mark.asyncio
    async def test_control_endpoints_write_events(lab,monkeypatch):
        import server
        monkeypatch.setattr(server,'lab',lab)
        async with httpx.AsyncClient(transport=httpx.ASGITransport(app=server.app),base_url='http://localhost:8020') as c:
            r=await c.post('/api/config',json={**lab.config,'concurrency':2})
            assert r.status_code==200
            r=await c.post('/api/toggle/agent/A8')
            assert r.status_code==200 and not lab.agents['A8'].up
            r=await c.get('/api/export')
            assert len(r.json()['events'])>=3
            r=await c.post('/api/config',

CalledProcessError: Command '['/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab/.venv/bin/python', '-m', 'pytest', '-q']' returned non-zero exit status 1.

## Recreate the environment (only if needed)
Homebrew installs the official vllm-metal package. The Python app dependencies stay in the project virtual environment. No cloud keys or services are needed.

In [ ]:
# Run deliberately on a fresh installation; not required on this prepared Mac.
# subprocess.run(["brew", "tap", "vllm-project/vllm-metal", "https://github.com/vllm-project/vllm-metal"], check=True)
# subprocess.run(["brew", "install", "vllm-project/vllm-metal/vllm-metal"], check=True, env={**os.environ, "HOMEBREW_NO_INSTALL_CLEANUP": "1"})
# subprocess.run(["uv", "venv", "--python", "3.12", ".venv"], check=True)
# subprocess.run(["uv", "pip", "install", "--python", ".venv/bin/python", "-r", "requirements.lock.txt"], check=True)

### traces.py

In [ ]:
%%writefile traces.py
"""Three-hour wall-clock retention. Cleanup runs even when simulation is paused."""
import json,time,uuid
TTL=3*60*60

class TraceStore:
    def __init__(self,db):
        self.db=db
        db.execute('PRAGMA secure_delete=ON')
        db.execute('CREATE TABLE IF NOT EXISTS traces(id TEXT PRIMARY KEY, run TEXT, kind TEXT, agent TEXT, created REAL, expires REAL, payload TEXT)')
        db.execute('CREATE INDEX IF NOT EXISTS trace_expiry ON traces(expires)')
        db.execute('CREATE INDEX IF NOT EXISTS event_wall ON events(wall)')
        db.commit();self.last_cleanup=0;self.purge()

    def create(self,run,kind,agent,**payload):
        now=time.time();identifier=uuid.uuid4().hex[:16]
        payload={'status':'pending','spans':[],**payload}
        self.db.execute('INSERT INTO traces VALUES(?,?,?,?,?,?,?)',(identifier,run,kind,agent,now,now+TTL,json.dumps(payload)))
        self.db.commit();return identifier

    def append(self,identifier,stage,**details):
        row=self.db.execute('SELECT payload FROM traces WHERE id=? AND expires>?',(identifier,time.time())).fetchone()
        if row is None:return
        data=json.loads(row[0]);data['spans'].append({'stage':stage,'wall':time.time(),**details})
        if 'status' in details:data['status']=details['status']
        if 'result' in details:data['result']=details['result']
        self.db.execute('UPDATE traces SET payload=? WHERE id=?',(json.dumps(data),identifier));self.db.commit()

    def get(self,identifier):
        row=self.db.execute('SELECT id,run,kind,agent,created,expires,payload FROM traces WHERE id=? AND expires>?',(identifier,time.time())).fetchone()
        if not row:return None
        return dict(zip(['id','run','kind','agent','created','expires','payload'],[*row[:6],json.loads(row[6])]))

    def recent(self,kind='',agent=''):
        query='SELECT id,run,kind,agent,created,expires,payload FROM traces WHERE expires>?';args=[time.time()]
        if kind:query+=' AND kind=?';args.append(kind)
        if agent:query+=' AND agent=?';args.append(agent)
        rows=self.db.execute(query+' ORDER BY created DESC LIMIT 100',args).fetchall()
        result=[]
        for identifier,run,kind,agent,created,expires,raw in rows:
            payload=json.loads(raw)
            result.append(dict(id=identifier,run=run,kind=kind,agent=agent,created=created,expires=expires,status=payload.get('status'),title=payload.get('title',kind)))
        return result

    def conversations(self,agent,direction='all'):
        rows=self.db.execute("SELECT id,run,kind,agent,created,expires,payload FROM traces WHERE expires>? AND kind IN ('message','debate') AND (agent=? OR json_extract(payload,'$.recipient')=? OR json_extract(payload,'$.result.target')=?) ORDER BY created DESC LIMIT 300",(time.time(),agent,agent,agent)).fetchall()
        items=[];linked=set()
        for identifier,run,kind,source,created,expires,raw in rows:
            data=json.loads(raw)
            if kind=='message' and data.get('model_trace_id'):linked.add(data['model_trace_id'])
        for identifier,run,kind,source,created,expires,raw in rows:
            data=json.loads(raw)
            if kind=='debate':
                if identifier in linked or not data.get('result'):continue
                result=data['result'];target=result.get('target');text=result.get('argument','');status='generated_only';trace=identifier
                reply=result.get('reply_to');evidence=result.get('evidence_id');caveat=result.get('uncertainty');message_kind='debate'
            else:
                target=data.get('recipient');text=data.get('text','');status=data.get('status');trace=data.get('model_trace_id') or identifier
                reply=data.get('reply_to');evidence=data.get('evidence_id');caveat=data.get('uncertainty');message_kind=data.get('message_kind')
            if direction=='sent' and source!=agent:continue
            if direction=='received' and (target!=agent or status!='delivered'):continue
            items.append(dict(id=identifier,run=run,source=source,recipient=target,text=text,status=status,kind=message_kind,created=created,expires=expires,trace_id=trace,reply_to=reply,evidence_id=evidence,uncertainty=caveat,direction='sent' if source==agent else 'received' if status=='delivered' else 'incoming_pending'))
        return items[:100]

    def purge(self,now=None):
        now=time.time() if now is None else now
        count=self.db.execute('DELETE FROM traces WHERE expires<=?',(now,)).rowcount
        count+=self.db.execute('DELETE FROM events WHERE wall<=?',(now-TTL,)).rowcount
        self.db.commit()
        self.db.execute('PRAGMA wal_checkpoint(TRUNCATE)')
        self.last_cleanup=now
        return count


### debate.py

In [ ]:
%%writefile debate.py
"""Evidence-bounded discussion, not a scientific fact-checking engine."""
import json,time
from collections import deque

TOPIC='Which path offers the most credible route to fault-tolerant quantum computing, and what experiment would distinguish the alternatives?'
CARDS=[
 {'id':'E1','title':'Surface-code memory below threshold','url':'https://www.nature.com/articles/s41586-024-08449-y','fact':'The Willow study reports below-threshold surface-code memories, including distance 7 and a distance-5 memory with a real-time decoder. This is evidence about memory; it does not alone demonstrate useful universal fault-tolerant algorithms.'},
 {'id':'E2','title':'High-threshold, low-overhead quantum memory','url':'https://www.nature.com/articles/s41586-024-07107-7','fact':'This study proposes a fault-tolerant memory protocol using bivariate bicycle quantum low-density parity-check (LDPC) codes. Its overhead and threshold results depend on the specified circuit noise model and connectivity assumptions; they are not an end-to-end useful quantum computer demonstration.'},
 {'id':'E3','title':'Logical processor using atom arrays','url':'https://www.nature.com/articles/s41586-023-06927-3','fact':'The reconfigurable neutral-atom study demonstrates logical operations using multiple encodings. Compare each reported operation and its error handling; do not equate a reported logical-qubit count with arbitrary long fault-tolerant computation.'},
 {'id':'E4','title':'Learning error decoding','url':'https://www.nature.com/articles/s41586-024-08148-8','fact':'This study develops a learned neural-network decoder for surface-code error correction. Decoder accuracy is one question; deployment latency, scaling and robustness must also be evaluated.'}
]
TEAMS=[('Surface codes','Argue for surface codes, but identify a limitation.','E1'),('Quantum LDPC','Argue for quantum LDPC, but expose connectivity assumptions.','E2'),('Neutral atoms','Argue for neutral-atom architectures, without overstating logical-qubit counts.','E3'),('Decoders & control','Examine decoding latency, calibration, leakage and control overhead.','E4'),('Critical reviewers','Compare evidence fairly; ask what experiment could falsify a claim.','E1')]
LENSES=['physical noise and correlations','logical memory versus logical gates','connectivity and control cost','decoding latency and scaling','resource estimates and falsifiable tests']
PHASES=['Opening claim','Cross-examination','Rebuttal','Synthesis']

class Debate:
    def __init__(self,lab):self.lab=lab;self.reset()
    def reset(self):
        self.turn=0;self.entries=deque(maxlen=150);self.outstanding=False;self.finished=False
    def role(self,aid):
        n=int(aid[1:])-1;team,brief,evidence=TEAMS[n//5]
        return dict(team=team,brief=brief,lens=LENSES[n%5],evidence=evidence)
    def observation(self,aid):
        role=self.role(aid);phase=PHASES[min(3,self.turn//25)]
        # A participant replies to a published earlier turn from a different team.
        others=[e for e in self.entries if e['team']!=role['team'] and not e.get('fallback')]
        previous=others[-1] if others else None
        recipient=previous['agent'] if previous else f'C{((int(aid[1:])-1+5)%25)+1:02d}'
        cards=[c for c in CARDS if c['id'] in {role['evidence'],'E1','E2'}][:3]
        phase_task={'Opening claim':'Make one bounded claim and propose a falsifiable test.','Cross-examination':'Challenge an assumption in the previous argument and ask one specific question.','Rebuttal':'Respond to the earlier argument; concede valid points and identify a remaining disagreement.','Synthesis':'Summarize a conditional conclusion, unresolved uncertainty and next experiment; do not declare a winner.'}[phase]
        return dict(mode='debate',agent=aid,topic=TOPIC,phase=phase,phase_task=phase_task,role=role,reply_to=previous['id'] if previous else None,recipient=recipient,previous_argument=previous['argument'] if previous else 'No prior argument yet.',evidence_cards=cards)
    def schedule(self):
        if self.outstanding or self.finished:return
        if self.turn>=100:self.finished=True;self.lab.running=False;return
        aid=f'C{self.turn%25+1:02d}'
        if not self.lab.agents[aid].up:self.turn+=1;return
        self.lab.pending[aid]=(time.monotonic(),self.lab.epoch,self.observation(aid));self.lab.agents[aid].status='queued';self.outstanding=True
    async def respond(self,aid,obs,wait):
        traces=self.lab.traces
        trace=traces.create(self.lab.run_id,'debate',aid,title=f"{obs['phase']} · {obs['role']['team']}",observation=obs,queue_seconds=wait)
        system='You are a scientific debate participant in an educational simulation. LDPC means low-density parity-check. Code distance is not called memory distance. A logical memory result is not universal fault-tolerant computation. Use only the supplied evidence cards for factual claims. Do not invent results or numbers. In 2-3 sentences, address the previous argument from your assigned lens, distinguish an observation from an assumption, and propose a discriminating test or limitation. Do not claim a scientific consensus or a winner. Return JSON with argument (under 100 words), evidence_id (one supplied ID), and uncertainty (one short caveat).'
        schema={'type':'object','properties':{'argument':{'type':'string'},'evidence_id':{'type':'string','enum':[c['id'] for c in obs['evidence_cards']]},'uncertainty':{'type':'string'}},'required':['argument','evidence_id','uncertainty'],'additionalProperties':False}
        messages=[{'role':'system','content':system},{'role':'user','content':json.dumps(obs)}]
        error='';start=time.monotonic()
        for attempt in range(2):
            body=None
            try:
                traces.append(trace,'inference_request',attempt=attempt+1,messages=messages,model=self.lab.model,max_tokens=256)
                response=await self.lab.client.post(self.lab.model_url+'/chat/completions',json=dict(model=self.lab.model,messages=messages,max_tokens=256,temperature=.4,chat_template_kwargs={'enable_thinking':False},response_format={'type':'json_schema','json_schema':{'name':'scientific_argument','strict':True,'schema':schema}}))
                response.raise_for_status();body=response.json()
                traces.append(trace,'inference_response',attempt=attempt+1,response=body)
                self.lab.tokens+=body.get('usage',{}).get('total_tokens',0)
                parsed=json.loads(body['choices'][0]['message']['content'])
                if not isinstance(parsed.get('argument'),str) or not parsed['argument'].strip() or parsed.get('evidence_id') not in [c['id'] for c in obs['evidence_cards']]:raise ValueError('Malformed argument or unknown evidence ID')
                if 'linear decoding' in parsed['argument'].lower() or 'memory distance' in parsed['argument'].lower():raise ValueError('Incorrect terminology: LDPC means low-density parity-check; use code distance.')
                result=dict(action='debate_message',target=obs['recipient'],reason=parsed['argument'],argument=parsed['argument'],evidence_id=parsed['evidence_id'],uncertainty=parsed.get('uncertainty',''),phase=obs['phase'],team=obs['role']['team'],reply_to=obs['reply_to'],trace_id=trace)
                traces.append(trace,'validated',status='validated',result=result,duration_seconds=time.monotonic()-start)
                return result,None
            except Exception as e:
                error=str(e)[:300]
                if isinstance(e,(ValueError,KeyError,TypeError)):self.lab.invalid+=1
                traces.append(trace,'error',attempt=attempt+1,error=error)
                messages.append({'role':'user','content':'Return valid JSON with argument, evidence_id and uncertainty. LDPC means low-density parity-check. Use code distance, not memory distance. Address your assigned team and lens; do not copy the earlier argument.'})
        result=dict(action='debate_message',target=obs['recipient'],reason='No valid model argument; retry this participant.',argument='No valid model argument was produced. No scientific claim is substituted.',evidence_id='',uncertainty=error,phase=obs['phase'],team=obs['role']['team'],reply_to=obs['reply_to'],trace_id=trace)
        traces.append(trace,'fallback',status='fallback',result=result);return result,error
    def apply(self,aid,result,error):
        entry=dict(id=f'T{self.turn+1}',agent=aid,wall=time.time(),**{k:result.get(k) for k in ['phase','team','argument','evidence_id','uncertainty','reply_to','trace_id']},recipient=result['target'],fallback=bool(error))
        self.entries.append(entry)
        self.lab.send(aid,result['target'],'debate',payload=entry)
        self.lab.log('debate',f"{aid} → {result['target']} · {result['phase']}: {result['argument']}",source=aid,trace_id=result['trace_id'])
        self.turn+=1;self.outstanding=False
        if self.turn>=100:self.finished=True;self.lab.running=False
    def snapshot(self):
        now=time.time();self.entries=deque((e for e in self.entries if now-e['wall']<10800),maxlen=150)
        return dict(topic=TOPIC,turn=self.turn,total_turns=100,phase=PHASES[min(3,self.turn//25)],finished=self.finished,entries=list(self.entries),evidence=CARDS,teams=[dict(name=t[0],brief=t[1]) for t in TEAMS])


### test_learning.py

In [ ]:
%%writefile test_learning.py
import asyncio,json,time
import httpx,pytest
from engine import Lab
from traces import TTL

@pytest.fixture
def lab(tmp_path):
    value=Lab(tmp_path/'learning.sqlite')
    yield value
    value.db.close()

def test_retention_is_wall_clock_and_across_runs(lab):
    expired=lab.traces.create('old-run','debate','C01',title='Old')
    current=lab.traces.create(lab.run_id,'request','C02',title='Current')
    lab.db.execute('UPDATE traces SET expires=? WHERE id=?',(time.time()-1,expired));lab.db.commit()
    assert lab.traces.get(expired) is None
    assert expired not in [t['id'] for t in lab.traces.recent()]
    lab.running=False;lab.cleanup()
    assert lab.db.execute('SELECT count(*) FROM traces WHERE id=?',(expired,)).fetchone()[0]==0
    assert lab.traces.get(current)
    assert round(lab.traces.get(current)['expires']-lab.traces.get(current)['created'])==10800

def test_network_trace_has_actual_task_and_service_result(lab):
    for _ in range(30):lab.step()
    complete=[t for t in lab.traces.recent('request') if t['status']=='complete']
    assert complete
    trace=lab.traces.get(complete[0]['id'])
    stages=[x['stage'] for x in trace['payload']['spans']]
    assert 'link_queued' in stages and 'service_result' in stages and 'response_received' in stages
    assert trace['payload']['result']['execution'].startswith('deterministic')

@pytest.mark.asyncio
async def test_debate_uses_shared_endpoint_and_records_prompt(lab):
    calls=[]
    def response(request):
        calls.append(request)
        return httpx.Response(200,json={'choices':[{'message':{'content':json.dumps({'argument':'Below-threshold memory is not universal logical computation. Test logical gates.','evidence_id':'E1','uncertainty':'Memory alone is insufficient.'})}}],'usage':{'total_tokens':100}})
    await lab.client.aclose();lab.client=httpx.AsyncClient(transport=httpx.MockTransport(response))
    lab.mode='debate';lab.debate.schedule()
    queued,epoch,obs=lab.pending.pop('C01')
    result,error=await lab.debate.respond('C01',obs,.1)
    assert error is None and str(calls[0].url).startswith(lab.model_url)
    lab.debate.apply('C01',result,error)
    assert lab.debate.turn==1 and lab.debate.entries[0]['evidence_id']=='E1'
    trace=lab.traces.get(result['trace_id'])
    assert any(x['stage']=='inference_request' and x['messages'][0]['role']=='system' for x in trace['payload']['spans'])
    assert any(x['stage']=='inference_response' for x in trace['payload']['spans'])
    assert lab.inflight # Actual debate message is queued through the simulated network.
    await lab.client.aclose()

def test_debate_assignments_and_round_limit(lab):
    assert len({lab.debate.role(f'C{i:02d}')['team'] for i in range(1,26)})==5
    lab.debate.turn=25;assert lab.debate.observation('C01')['phase']=='Cross-examination'
    lab.debate.turn=100;lab.debate.schedule()
    assert lab.debate.finished and not lab.running

@pytest.mark.asyncio
async def test_trace_api_expiry(lab,monkeypatch):
    import server
    monkeypatch.setattr(server,'lab',lab)
    identifier=lab.traces.create(lab.run_id,'inference','C01',title='Example')
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=server.app),base_url='http://localhost:8020') as c:
        assert (await c.get('/api/traces/'+identifier)).status_code==200
        lab.db.execute('UPDATE traces SET expires=? WHERE id=?',(time.time()-1,identifier));lab.db.commit()
        assert (await c.get('/api/traces/'+identifier)).status_code==404
        assert (await c.get('/api/traces')).json()['items']==[]
        assert (await c.post('/api/mode/debate')).json()['mode']=='debate'
    await lab.client.aclose()

def test_agent_conversation_delivery_and_direction(lab):
    lab.send('C01','C06','debate',payload={'argument':'Test the logical gate error rate.','reply_to':'T1','evidence_id':'E1'})
    sent=lab.traces.conversations('C01','sent')
    assert len(sent)==1 and sent[0]['recipient']=='C06'
    assert lab.traces.conversations('C06','received')==[]
    for _ in range(5):lab.step()
    received=lab.traces.conversations('C06','received')
    assert len(received)==1 and received[0]['source']=='C01'
    assert received[0]['text']=='Test the logical gate error rate.'
    assert received[0]['status']=='delivered'

def test_conversation_loss_not_marked_delivered(lab):
    lab.config['loss']=100
    lab.send('C01','A4','status',payload={'note':'Ready'})
    for _ in range(3):lab.step()
    assert lab.traces.conversations('A4','received')==[]
    assert lab.traces.conversations('C01','sent')[0]['status']=='dropped'

def test_expired_conversations_hidden(lab):
    lab.send('C01','A4','status',payload={'note':'Temporary'})
    message=lab.traces.conversations('C01')[0]
    lab.db.execute('UPDATE traces SET expires=? WHERE id=?',(time.time()-1,message['id']));lab.db.commit()
    assert lab.traces.conversations('C01')==[]


## Current 100-client implementation\nThe authoritative runnable source is in this same folder. The current network and scientific-debate modes both create `C01` through `C100` (100 clients), with 20 shared inference workers and one vLLM-metal model. Use the next cell to inspect the live source without overwriting it.

In [ ]:
from pathlib import Path\nroot=Path.cwd()\nif not (root/'engine.py').exists(): root=Path('/Users/dheemanth/Documents/Codex/2026-10-18/a/outputs/agent-network-lab')\nprint('Clients: C01–C100 (100)')\nprint('Shared inference concurrency: 20')\nfor name in ['engine.py','debate.py','server.py','manage.py','static/index.html']:\n    print(f'\n===== {name} =====')\n    print((root/name).read_text()[:1200])